# Day 030 Project Solution — Workflow Orchestration

A `Pipeline` that chains multiple automation steps, captures success and failure, and generates an AI-powered run report.

In [ ]:
import ollama
import time


import time


def run_step(name: str, fn) -> dict:
    start = time.time()
    try:
        result = fn()
        return {
            "name":       name,
            "status":     "ok",
            "result":     result,
            "error":      None,
            "duration_s": round(time.time() - start, 3),
        }
    except Exception as e:
        return {
            "name":       name,
            "status":     "error",
            "result":     None,
            "error":      str(e),
            "duration_s": round(time.time() - start, 3),
        }


def chain_steps(steps: list, stop_on_error: bool = True) -> list:
    results = []
    failed  = False
    for name, fn in steps:
        if failed and stop_on_error:
            results.append({
                "name":       name,
                "status":     "skipped",
                "result":     None,
                "error":      None,
                "duration_s": 0.0,
            })
        else:
            step_result = run_step(name, fn)
            results.append(step_result)
            if step_result["status"] == "error":
                failed = True
    return results


def summarize_run(step_results: list) -> dict:
    statuses = [s["status"] for s in step_results]
    return {
        "total":            len(step_results),
        "passed":           statuses.count("ok"),
        "failed":           statuses.count("error"),
        "skipped":          statuses.count("skipped"),
        "total_duration_s": round(
            sum(s.get("duration_s", 0.0) for s in step_results), 3
        ),
        "all_ok":           all(s == "ok" for s in statuses),
    }


def ai_pipeline_summary(step_results: list, model: str = "llama3.2") -> str:
    summary = summarize_run(step_results)
    lines = []
    for s in step_results:
        if s["status"] == "ok":
            lines.append(f"  \u2713 {s['name']} ({s['duration_s']:.3f}s)")
        elif s["status"] == "error":
            lines.append(f"  \u2717 {s['name']}: {s['error']}")
        else:
            lines.append(f"  - {s['name']}: skipped")
    run_text = (
        f"{summary['passed']}/{summary['total']} steps passed, "
        f"{summary['total_duration_s']}s total\n"
        + "\n".join(lines)
    )
    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a pipeline monitor. "
                    "Summarise a workflow run in 2\u20133 sentences. Be concise."
                ),
            },
            {
                "role": "user",
                "content": f"{run_text}\n\nSummarise the run:",
            },
        ],
    )
    return response["message"]["content"]


class Pipeline:
    def __init__(
        self,
        name: str = "pipeline",
        stop_on_error: bool = True,
        model: str = "llama3.2",
    ):
        self.name          = name
        self.stop_on_error = stop_on_error
        self.model         = model
        self._steps: list  = []

    def add_step(self, name: str, fn) -> "Pipeline":
        self._steps.append((name, fn))
        return self

    def run(self) -> dict:
        step_results = chain_steps(self._steps, stop_on_error=self.stop_on_error)
        summary      = summarize_run(step_results)
        report       = ai_pipeline_summary(step_results, model=self.model)
        return {
            "name":    self.name,
            "steps":   step_results,
            "summary": summary,
            "report":  report,
        }

## Action 1 — Run a Successful Three-Step Pipeline

In [ ]:
p = Pipeline(name='daily_digest', stop_on_error=True)
p.add_step('fetch_articles',   lambda: {'articles': 12, 'sources': 3})
p.add_step('process_content',  lambda: 'Processed 12 articles into 8 summaries')
p.add_step('generate_report',  lambda: '/tmp/day030_report.txt')

result = p.run()
print(f"Pipeline : {result['name']}")
print(f"Total    : {result['summary']['total']} steps")
print(f"Passed   : {result['summary']['passed']}")
print(f"All OK   : {result['summary']['all_ok']}")
print(f"Duration : {result['summary']['total_duration_s']}s")

## Action 2 — Demonstrate Stop-on-Error with a Failing Step

In [ ]:
def _fail_step():
    raise ValueError('simulated network error')

p2 = Pipeline(name='with_failure', stop_on_error=True)
p2.add_step('prepare',   lambda: 'ready')
p2.add_step('fetch_remote', _fail_step)
p2.add_step('post_process', lambda: 'this is skipped')

result2 = p2.run()
for s in result2['steps']:
    print(f"  {s['status']:8} {s['name']}"
          + (f" — {s['error']}" if s['error'] else ''))
print(f"\nFailed : {result2['summary']['failed']}")
print(f"Skipped: {result2['summary']['skipped']}")
print(f"All OK : {result2['summary']['all_ok']}")

## Action 3 — AI Run Report and Verify

In [ ]:
# Re-use the successful pipeline result from Action 1
print('AI Run Report:')
print(result['report'])

# Confirm structure
assert result['summary']['all_ok'] is True
assert len(result['steps']) == 3
assert isinstance(result['report'], str) and len(result['report']) > 10

print('\nOrchestration complete!')